# Data Cleaning Step 2: Categorical Encoding

**Objective:** 
1. Identify all categorical columns (object type).
2. Extract unique categories for each column.
3. Create a mapping table (Category -> Number).
4. Convert categorical columns to numeric codes based on the mapping.
   - Note: We will preserve the original logic (e.g., "1. Agree" -> 1).

**Input File:** `TIGPS_W2_studentdata_ver1_missing_handled.csv`
**Output Files:** 
- Mapping Table: `category_mapping_table.csv`
- Encoded Data: `TIGPS_W2_studentdata_ver2_encoded.csv`

In [4]:
import pandas as pd
import numpy as np
import re

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Data

In [5]:
# Using ver1 as input based on availability
input_path = r"..\..\Data\2024data\TIGPS_W2_studentdata_ver2_missing_handled.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Data loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: File not found at {input_path}")

Data loaded successfully. Shape: (8892, 363)


## 2. Identify Categorical Columns

We will look for columns of type `object`.

In [6]:
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Number of categorical columns: {len(cat_cols)}")
# print(cat_cols)

Number of categorical columns: 327


## 3. Generate Mapping and Encode

We will iterate through each categorical column, find unique values, and create a mapping.
Strategy:
- If the string starts with a number (e.g., "1. Agree"), we extract that number.
- If it's a pure string category (e.g., "Male", "Female"), we assign a code (1, 2...).
- We need to be careful with "Unknown" or other placeholders if they exist.

In [7]:
mapping_records = []
df_encoded = df.copy()

def extract_number(text):
    """Extracts the leading number from a string like '1. Agree'. Returns None if no number found."""
    if not isinstance(text, str):
        return None
    match = re.match(r"^(-?\d+)", text.strip())
    if match:
        return int(match.group(1))
    # Handle simplified format "1" or "1."
    match_dot = re.match(r"^(-?\d+)\.", text.strip())
    if match_dot:
        return int(match_dot.group(1))
    return None

for col in cat_cols:
    # Skip identifier columns if necessary (like name, email, cell) - Add logic if user wants to keep them as strings
    # For this specific dataset, most 'object' columns are questionnaire items.
    # We will try to convert everything, but keep a note for high-cardinality columns (like names).
    
    unique_vals = df[col].dropna().unique()
    
    # Heuristic: If too many unique values (e.g. > 50), it might be an open text field or ID.
    # Check if they look like Likert scale items (start with number).
    is_likert = all([extract_number(str(v)) is not None for v in unique_vals if str(v).strip() != ''])
    
    if len(unique_vals) > 50 and not is_likert:
        print(f"Skipping likely text/ID column: {col} (Unique values: {len(unique_vals)})")
        continue
        
    for val in unique_vals:
        original_text = str(val)
        
        # Determine numeric code
        code = extract_number(original_text)
        
        if code is None:
            # Assign a new code if no leading number found
            # Strategy: Sort unique values and assign 1-based index?
            # Or manually map? For automation, we might need a fallback.
            # Let's check if we can just enumerate them in order.
            continue # Only doing extraction for now as requested for specific format items
        
        mapping_records.append({
            'Column': col,
            'Original_Value': original_text,
            'Mapped_Code': code
        })
        
        # Apply encoding (Specific value replacement to avoid partial matches)
        # Using mask for safety
        mask = df_encoded[col] == val
        df_encoded.loc[mask, col] = code

print("Encoding loop definition complete. Ready to run logic.")

Skipping likely text/ID column: student_oid (Unique values: 8892)
Skipping likely text/ID column: student_id (Unique values: 8892)
Skipping likely text/ID column: school_name (Unique values: 172)
Skipping likely text/ID column: class (Unique values: 103)
Skipping likely text/ID column: name (Unique values: 8583)
Skipping likely text/ID column: cell (Unique values: 8663)
Skipping likely text/ID column: email (Unique values: 8478)
Encoding loop definition complete. Ready to run logic.


In [8]:
# Execute the loop logic properly (re-writing for execution flow)

mapping_list = []
cols_processed = 0

for col in cat_cols:
    unique_vals = df[col].dropna().unique()
    
    # Skip high cardinality non-likert (likely IDs or open text)
    if len(unique_vals) > 100: 
        # Double check if it's numeric-ish
        first_val = str(unique_vals[0])
        if not(first_val[0].isdigit() or first_val.startswith('-')):
             continue

    # Create a mapping dictionary for this column
    col_map = {}
    for val in unique_vals:
        text = str(val)
        code = extract_number(text)
        if code is not None:
            col_map[val] = code
            mapping_list.append({'Column': col, 'Original_Value': text, 'Mapped_Code': code})
        else:
            # Handling pure text categories (like 'Male', 'Female')
            # We need a stable assignment. Let's use sorted order index + 1.
            # NOTE: This part requires caution. If 'Male' gets 1 and 'Female' 2, record it.
            pass 
    
    # Apply map if not empty
    if col_map:
        df_encoded[col] = df_encoded[col].map(col_map).fillna(df_encoded[col]) # Keep NaNs/Unmapped as is
        cols_processed += 1

print(f"Processed {cols_processed} categorical columns.")

C:\Users\user\AppData\Local\Temp\ipykernel_132616\2662875243.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded[col] = df_encoded[col].map(col_map).fillna(df_encoded[col]) # Keep NaNs/Unmapped as is
C:\Users\user\AppData\Local\Temp\ipykernel_132616\2662875243.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded[col] = df_encoded[col].map(col_map).fillna(df_encoded[col]) # Keep NaNs/Unmapped as is
C:\Users\user\AppData\Local\Temp\ipykernel_132616\2662875243.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .

Processed 321 categorical columns.


C:\Users\user\AppData\Local\Temp\ipykernel_132616\2662875243.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded[col] = df_encoded[col].map(col_map).fillna(df_encoded[col]) # Keep NaNs/Unmapped as is
C:\Users\user\AppData\Local\Temp\ipykernel_132616\2662875243.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_encoded[col] = df_encoded[col].map(col_map).fillna(df_encoded[col]) # Keep NaNs/Unmapped as is
C:\Users\user\AppData\Local\Temp\ipykernel_132616\2662875243.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .

## 4. Save Outputs

In [9]:
# Save Mapping Table
mapping_df = pd.DataFrame(mapping_list)
mapping_df = mapping_df.sort_values(by=['Column', 'Mapped_Code'])
mapping_df.to_csv("category_mapping_table.csv", index=False, encoding='utf-8-sig')
print("Mapping table saved to 'category_mapping_table.csv'.")

# Convert processed columns to numeric where possible to finalize schema
for col in df_encoded.columns:
    try:
        df_encoded[col] = pd.to_numeric(df_encoded[col])
    except:
        pass # Keep as object if it still contains non-numeric strings

# Save Encoded Data
output_file_encoded = r"..\..\Data\2024data\TIGPS_W2_studentdata_ver2_encoded.csv"
df_encoded.to_csv(output_file_encoded, index=False, encoding='utf-8-sig')
print(f"Encoded data saved to {output_file_encoded}")

Mapping table saved to 'category_mapping_table.csv'.
Encoded data saved to ..\..\Data\2024data\TIGPS_W2_studentdata_ver2_encoded.csv
